## No memory

In [1]:
%run init_env.py

Added to Python path: C:\git\cs5305
Environment initialization completed successfully.


In [2]:
llm = create_azure_llm()
agent = create_agent(llm)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [3]:
question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pprint(response["messages"][-1].content)

("Hello Ali! It's nice to meet you. Green is a wonderful color—so refreshing "
 'and calming. How can I assist you today?')


In [4]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response["messages"][-1].content)

"I don't know your favourite colour yet. Would you like to tell me what it is?"


## Memory

In [5]:
# langgraph checkpoint API.  Snapshot of the graph state at a given point in time. 
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),  
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [7]:
pprint(response["messages"][-1].content)

("Hello Ali! It's great to meet you. Green is a wonderful color—so refreshing "
 'and calming. How can I assist you today?')


In [8]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response["messages"][-1].content)

'Your favourite colour is green!'


In [9]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pprint(response["messages"][-1].content)

# Optional: inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pprint(state.values["messages"][-4:])

('According to your saved profile, you live in Lahore and your loyalty tier is '
 'gold. How can I assist you further, Ali?')
[HumanMessage(content="Store this custom data in memory and use it for future answers:\n{'name': 'Ali', 'favorite_color': 'green', 'city': 'Lahore', 'loyalty_tier': 'gold'}", additional_kwargs={}, response_metadata={}, id='0f81a966-b2c1-4d9c-8b10-027c9462a3f8'),
 AIMessage(content="Got it, Ali! I've stored your information: your name is Ali, your favorite color is green, you live in Lahore, and your loyalty tier is gold. I'll use this information to personalize our future conversations. How can I assist you next?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 116, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

In [10]:
def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    return result["messages"][-1].content

In [11]:

# Scenario 1: Thread isolation (different users/sessions)
config_a = {"configurable": {"thread_id": "memory-demo-A"}}
config_b = {"configurable": {"thread_id": "memory-demo-B"}}

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

print("Scenario 1A:", ask(config_a, "What database do I prefer?"))
print("Scenario 1B:", ask(config_b, "What database do I prefer?"))

Scenario 1A: You prefer PostgreSQL as your database.
Scenario 1B: You prefer MongoDB.


In [12]:
# Scenario 2: Preference update within the same thread
config_update = {"configurable": {"thread_id": "memory-demo-update"}}

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
print("Scenario 2:", ask(config_update, "Which deployment region should you use now?"))

# Scenario 3: Multi-step task memory
config_task = {"configurable": {"thread_id": "memory-demo-task"}}

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
print("Scenario 3A:", ask(config_task, "Summarize my constraints in one sentence."))
print("Scenario 3B:", ask(config_task, "Given those constraints, suggest 3 practical next steps."))

# Optional: inspect latest stored messages for one scenario
task_state = agent.get_state(config_task)
print("\nRecent messages in Scenario 3 thread:")
pprint(task_state.values["messages"][-4:])

Scenario 2: Based on your update, the deployment region you should use now is **West Europe**.
Scenario 3A: You need to build a functional chatbot MVP by Friday with a budget of $2,000.
Scenario 3B: 1. **Select a no-code chatbot platform** (like Tars, Landbot, or Dialogflow) that fits your budget and supports your target channels to speed up development.  
2. **Outline the chatbot’s main conversation flows and key features**, focusing only on essential functionality to meet the MVP deadline.  
3. **Set up and test the chatbot internally by Thursday**, allowing time for quick revisions before deployment on Friday.

Recent messages in Scenario 3 thread:
[HumanMessage(content='Summarize my constraints in one sentence.', additional_kwargs={}, response_metadata={}, id='04cc8405-02bc-4f47-a0f6-c2e36ad7f157'),
 AIMessage(content='You need to build a functional chatbot MVP by Friday with a budget of $2,000.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_t